In [104]:
import pandas as pd

In [105]:
df_metadata = pd.read_csv("../data/DOI-10-13012-b2idb-4610831_v3/Willoughby2014_286citing_publications.csv")

In [106]:
# drop the two publications that are chinese
drop_dois = ['10.13220/j.cnki.jipr.2015.06.004',
             '10.19540/j.cnki.cjcmm.20200604.201']

df_metadata = df_metadata[~df_metadata['DOI'].isin(drop_dois)]
print(len(df_metadata))

284


In [107]:
df_cc = pd.read_csv("../data/DOI-10-13012-b2idb-4610831_v3/Willoughby2014_citation_contexts.csv")

In [108]:
df_cc.columns

Index(['DOI',
       'Is it a review according to Scopus? (retrieval date: November 25, 2023)',
       'Is it a review according to WoS? (retrieval date: November 25, 2023)',
       'Do at least one of Neupane et al., 2019 or Willoughby et al., 2020 appear in the bibliography according to Scopus?',
       'Do at least one of Neupane et al., 2019 or Willoughby et al., 2020 appear in the bibliography according to WoS?',
       'Does the introduction section contain all the citations to Willoughby et al., 2014?',
       'label in the reference list', 'Number of citation contexts',
       'Citation marker', 'Citation context', 'Other notes'],
      dtype='str')

In [109]:
# DOI to lower case (normalization)
norm = lambda s: s.str.strip().str.lower()
df_cc["DOI"] = norm(df_cc["DOI"])
df_metadata["DOI"] = norm(df_metadata["DOI"])
df_cc["cc_id"] = df_cc.groupby("DOI").cumcount() + 1

# create paper-level table (one row per DOI) vs. context-level table
paper_cols = [c for c in df_cc.columns[0:6]]
df_paper = df_cc[paper_cols]


In [110]:
# check the inconsistency in the df_cc table, in particular, one DOI has different answers to paper_cols
conflict = df_paper.groupby("DOI").nunique(dropna=False).gt(1)
print(conflict.sum())

Is it a review according to Scopus? (retrieval date: November 25, 2023)                                                0
Is it a review according to WoS? (retrieval date: November 25, 2023)                                                   0
Do at least one of Neupane et al., 2019 or Willoughby et al., 2020 appear in the bibliography according to Scopus?     0
Do at least one of Neupane et al., 2019 or Willoughby et al., 2020 appear in the bibliography according to WoS?        0
Does the introduction section contain all the citations to Willoughby et al., 2014?                                   13
dtype: int64


In [111]:
# drop those 13 DOIs with conflicting answers
n = df_paper.groupby("DOI")['Does the introduction section contain all the citations to Willoughby et al., 2014?'].nunique(dropna=False)
bad_dois = n[n > 1].index
print(len(bad_dois))
print(bad_dois)# should print 13

13
Index(['10.1002/mrc.4951', '10.1016/j.comptc.2014.11.007',
       '10.1016/j.molstruc.2017.09.058', '10.1016/j.phytochem.2022.113200',
       '10.1021/acs.jctc.7b00772', '10.1021/acs.joc.5b02396',
       '10.1021/acs.joc.8b02975', '10.1021/acs.jpca.7b01954',
       '10.1039/d2nj02587f', '10.1186/s13321-022-00587-7',
       '10.3389/fmolb.2021.756219', '10.3390/cryst13010016',
       '10.3390/md15060171'],
      dtype='str', name='DOI')


In [112]:
# Keep only rows whose DOI is not in that list
df_paper_clean = df_paper[~df_paper["DOI"].isin(bad_dois)]

In [113]:
# Recheck conflicts
conflict = df_paper_clean.groupby("DOI").nunique(dropna=False).gt(1)
print(conflict.sum())

Is it a review according to Scopus? (retrieval date: November 25, 2023)                                               0
Is it a review according to WoS? (retrieval date: November 25, 2023)                                                  0
Do at least one of Neupane et al., 2019 or Willoughby et al., 2020 appear in the bibliography according to Scopus?    0
Do at least one of Neupane et al., 2019 or Willoughby et al., 2020 appear in the bibliography according to WoS?       0
Does the introduction section contain all the citations to Willoughby et al., 2014?                                   0
dtype: int64


In [114]:
# regroup
df_paper_clean = df_paper_clean.groupby('DOI').first().reset_index()
len(df_paper_clean)

271

In [115]:
df_metadata.columns

Index(['Item Type', 'Publication Year', 'Author', 'Title', 'Publication Title',
       'ISBN', 'ISSN', 'DOI', 'Url', 'Abstract', 'Date', 'Pages', 'Issue',
       'Volume', 'Journal Abbreviation', 'Short Title', 'Language',
       'Library Catalog', 'Conference Name'],
      dtype='str')

In [116]:
df_merged = df_paper_clean.merge(df_metadata[['DOI', 'Author', 'Title', 'Date']], on="DOI", how="left")
df_merged.head()

,DOI,"Is it a review according to Scopus? (retrieval date: November 25, 2023)","Is it a review according to WoS? (retrieval date: November 25, 2023)","Do at least one of Neupane et al., 2019 or Willoughby et al., 2020 appear in the bibliography according to Scopus?","Do at least one of Neupane et al., 2019 or Willoughby et al., 2020 appear in the bibliography according to WoS?","Does the introduction section contain all the citations to Willoughby et al., 2014?",Author,Title,Date
0,10.1002/anie.201706532,n,n,n,n,n,"Matsuo, Yosuke; Yoshida, Ayane; Saito, Yoshino...",Structural Revision and Biomimetic Synthesis o...,2017
1,10.1002/anie.201708266,n,n,n,n,n,"Grimme, Stefan; Bannwarth, Christoph; Dohm, Se...",Fully Automated Quantum‐Chemistry‐Based Comput...,11/13/17
2,10.1002/anie.201810566,n,n,n,n,n,"Elkin, Masha; Scruse, Anthony C.; Turlik, Anet...",Computational and Synthetic Investigation of C...,1/21/19
3,10.1002/anie.201902777,n,n,n,n,n,"Kutateladze, Andrei G.; Krenske, Elizabeth H.;...",Reassignments and Corroborations of Oxo‐Bridge...,5/20/19
4,10.1002/anie.201916566,n,n,n,n,n,"Seeman, Jeffrey I.; Tantillo, Dean J.",From Decades to Minutes: Steps Toward the Stru...,6/26/20


In [117]:
len(df_merged)

271

In [118]:
results = pd.DataFrame()

In [119]:
results['DOI'] = df_merged['DOI']

In [120]:
results['Aware'] = (
    df_merged[["Is it a review according to Scopus? (retrieval date: November 25, 2023)",
        "Is it a review according to WoS? (retrieval date: November 25, 2023)"]]
    .eq("y")
    .any(axis=1)
    .map({True: "y", False: "n"})
)

In [121]:
results['Review'] = (
    df_merged[["Is it a review according to Scopus? (retrieval date: November 25, 2023)",
        "Is it a review according to WoS? (retrieval date: November 25, 2023)"]]
    .eq("y")
    .any(axis=1)
    .map({True: "y", False: "n"})
)

In [122]:
results['IntroOnly'] = df_merged['Does the introduction section contain all the citations to Willoughby et al., 2014?']

In [123]:
# creating tag
author = (df_merged["Author"]
          .str.split(";").str[0]      # first author
          .str.split(",").str[0]      # last name
          .str.strip().str.lower())

year = df_merged["Date"].astype(str).str.split("-").str[0]

title = (df_merged["Title"]
         .str.split(" ").str[0]       # first word
         .str.lower())

results["tag"] = author + "_" + year + "_" + title

In [124]:
template_awareness = """Publication {tag} (DOI: {DOI}) is classified as not at risk of unreliability propagation because it cites either the paper reporting the code glitch (Neupane et al., 2019) or the addendum (Willoughby et al., 2020). Such a citation suggests that the authors were aware of the glitch and were therefore unlikely to carry its effects into their own work. This inference is defeasible: further evidence could lead us to revise the classification."""

template_review = """Publication {tag} (DOI: {DOI}) is classified as not at risk of unreliability propagation because it is a review article. Review articles generally discuss existing research rather than execute computational procedures such as the WJH protocol. Review status therefore provides a reason not to infer that the authors used the protocol merely because they cited it."""

template_intro_only = """Publication {tag} (DOI: {DOI}) is classified as not at risk of unreliability propagation because all citations to the WJH protocol appear in the introduction. This suggests that the protocol is cited as background information rather than used to conduct the research. However, this inference is defeasible: we identified two counterexamples previously in which all citations appeared in the introduction, but the citation text indicated use of the protocol."""

In [125]:
# The message to use for each column
templates = {
    "Aware":    template_awareness,
    "Review":  template_review,
    "IntroOnly": template_intro_only
}

results["message"] = results.apply(
    lambda row: [tpl.format(**row) for col, tpl in templates.items() if row[col] == "y"],
    axis=1,
)

In [132]:
len(results)

271

In [131]:
len(df_metadata)

284